# Bangalore House Price Prediction – Updated End-to-End Notebook

This notebook is an improved version of the original BHP project. It preserves the core workflow:

- Data loading and inspection
- Missing-value handling
- Feature engineering
- Location grouping
- Outlier removal
- One-hot encoding and scaling
- Regression modelling
- Flask-ready model serialization

It additionally improves the original workflow with:

- Visual EDA
- Explicit handling of invalid `total_sqft` values
- 5-fold cross-validation
- Hyperparameter tuning with `GridSearchCV`
- Evaluation using R², MAE and RMSE
- Automatic best-model selection based on cross-validation performance


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import pickle
import warnings

warnings.filterwarnings("ignore")

## 2. Load the Dataset

In [ ]:
DATA_PATH = "Bengaluru_House_Data.csv"

data = pd.read_csv(DATA_PATH)

print("Dataset Shape:", data.shape)
data.head()

## 3. Initial Data Inspection

In [ ]:
print("First five rows:")
display(data.head())

print("\nLast five rows:")
display(data.tail())

print("\nDataset information:")
data.info()

print("\nMissing values:")
display(data.isnull().sum().sort_values(ascending=False))

print("\nDescriptive statistics:")
display(data.describe(include="all"))

## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values

In [ ]:
missing_values = data.isnull().sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(missing_values.index, missing_values.values)
plt.title("Missing Values in Each Column")
plt.xlabel("Columns")
plt.ylabel("Missing Value Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.2 House Price Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(data["price"].dropna(), bins=50)
plt.title("Distribution of House Prices")
plt.xlabel("Price (Lakhs)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 4.3 Top Locations by Number of Listings

In [ ]:
top_locations = data["location"].value_counts().head(10).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(top_locations.index, top_locations.values)
plt.title("Top 10 Locations by Listing Count")
plt.xlabel("Number of Listings")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

## 5. Remove Unnecessary Columns

In [ ]:
columns_to_drop = [
    "area_type",
    "availability",
    "society",
    "balcony"
]

data = data.drop(columns=columns_to_drop)

print("Remaining columns:")
print(data.columns.tolist())

print("\nNew dataset shape:", data.shape)

## 6. Handle Missing Values

In [ ]:
# Categorical columns: use the most frequent value
data["location"] = data["location"].fillna(data["location"].mode()[0])
data["size"] = data["size"].fillna(data["size"].mode()[0])

# Numerical column: use median
data["bath"] = data["bath"].fillna(data["bath"].median())

print("Missing values after initial handling:")
print(data.isnull().sum())

## 7. Feature Engineering

### 7.1 Extract BHK from the `size` Column

In [ ]:
data["bhk"] = data["size"].str.split().str[0].astype(int)

display(data[["size", "bhk"]].head())

### 7.2 Convert `total_sqft` into Numeric Values

In [ ]:
def convert_range(value):
    value = str(value).strip()

    # Handle ranges such as 1000-1200
    parts = value.split("-")

    if len(parts) == 2:
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan

    # Handle normal numeric values
    try:
        return float(value)
    except ValueError:
        return np.nan


data["total_sqft"] = data["total_sqft"].apply(convert_range)

print("Invalid / non-convertible total_sqft values:",
      data["total_sqft"].isna().sum())

# Explicitly remove rows whose total_sqft could not be converted
data = data.dropna(subset=["total_sqft"]).copy()

print("Rows after removing invalid total_sqft values:", data.shape[0])
print("Remaining missing total_sqft values:", data["total_sqft"].isna().sum())

### 7.3 BHK Distribution

In [ ]:
bhk_counts = data["bhk"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
plt.bar(bhk_counts.index.astype(str), bhk_counts.values)
plt.title("Distribution of BHK")
plt.xlabel("BHK")
plt.ylabel("Number of Properties")
plt.tight_layout()
plt.show()

### 7.4 Price vs Total Square Feet

In [ ]:
# Clip only for visualisation so extreme values do not dominate the chart
plot_data = data[
    data["total_sqft"] <= data["total_sqft"].quantile(0.99)
]

plt.figure(figsize=(10, 6))
plt.scatter(plot_data["total_sqft"], plot_data["price"], alpha=0.4)
plt.title("Total Square Feet vs Price")
plt.xlabel("Total Square Feet")
plt.ylabel("Price (Lakhs)")
plt.tight_layout()
plt.show()

### 7.5 Create Price per Square Foot

In [ ]:
data["price_per_sqft"] = (
    data["price"] * 100000 / data["total_sqft"]
)

display(data.head())

## 8. Location Processing

### 8.1 Remove Extra Spaces

In [ ]:
data["location"] = data["location"].astype(str).str.strip()

### 8.2 Group Rare Locations into `other`

In [ ]:
location_counts = data["location"].value_counts()

rare_locations = location_counts[
    location_counts <= 10
].index

data["location"] = data["location"].apply(
    lambda location: "other"
    if location in rare_locations
    else location
)

print("Number of unique locations after grouping:",
      data["location"].nunique())

print(data["location"].value_counts().head(10))

## 9. Outlier Removal

### 9.1 Remove Properties with Unrealistically Low Area per BHK

In [ ]:
rows_before_sqft_filter = data.shape[0]

data = data[
    (data["total_sqft"] / data["bhk"]) >= 300
].copy()

print("Rows before filter:", rows_before_sqft_filter)
print("Rows after filter:", data.shape[0])

### 9.2 Remove Location-Wise Price per Square Foot Outliers

In [ ]:
def remove_price_per_sqft_outliers(df):
    cleaned_groups = []

    for location, location_df in df.groupby("location"):
        mean = location_df["price_per_sqft"].mean()
        std = location_df["price_per_sqft"].std()

        filtered = location_df[
            (location_df["price_per_sqft"] > (mean - std)) &
            (location_df["price_per_sqft"] <= (mean + std))
        ]

        cleaned_groups.append(filtered)

    return pd.concat(cleaned_groups, ignore_index=True)


rows_before_price_filter = data.shape[0]

data = remove_price_per_sqft_outliers(data)

print("Rows before location-wise price filter:", rows_before_price_filter)
print("Rows after location-wise price filter:", data.shape[0])

### 9.3 Remove BHK Price Inconsistencies

In [ ]:
def remove_bhk_outliers(df):
    exclude_indices = []

    for location, location_df in df.groupby("location"):
        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby("bhk"):
            bhk_stats[bhk] = {
                "mean": bhk_df["price_per_sqft"].mean(),
                "count": bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby("bhk"):
            previous_bhk_stats = bhk_stats.get(bhk - 1)

            if previous_bhk_stats and previous_bhk_stats["count"] > 5:
                invalid_rows = bhk_df[
                    bhk_df["price_per_sqft"]
                    < previous_bhk_stats["mean"]
                ]

                exclude_indices.extend(invalid_rows.index.tolist())

    return df.drop(index=exclude_indices)


rows_before_bhk_filter = data.shape[0]

data = remove_bhk_outliers(data)

print("Rows before BHK consistency filter:", rows_before_bhk_filter)
print("Rows after BHK consistency filter:", data.shape[0])

## 10. Visualise the Data After Cleaning

### 10.1 Cleaning Pipeline Row Reduction

In [ ]:
stages = [
    "Original",
    "After invalid sqft removal",
    "After sqft/BHK rule",
    "After price/sqft rule",
    "After BHK rule"
]

# The first two counts are reconstructed from the current workflow
# for visual comparison.
stage_counts = [
    pd.read_csv(DATA_PATH).shape[0],
    rows_before_sqft_filter,
    rows_before_price_filter,
    rows_before_bhk_filter,
    data.shape[0]
]

plt.figure(figsize=(10, 5))
plt.plot(stages, stage_counts, marker="o")
plt.title("Record Count Through Data Cleaning")
plt.xlabel("Cleaning Stage")
plt.ylabel("Number of Rows")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 10.2 Correlation Heatmap

In [ ]:
numeric_columns = [
    "total_sqft",
    "bath",
    "bhk",
    "price"
]

correlation_matrix = data[numeric_columns].corr()

plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar()

plt.xticks(
    range(len(numeric_columns)),
    numeric_columns,
    rotation=45
)

plt.yticks(
    range(len(numeric_columns)),
    numeric_columns
)

for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        plt.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

## 11. Create the Final Modelling Dataset

In [ ]:
model_data = data.drop(
    columns=["size", "price_per_sqft"]
).copy()

# Final safety check
model_data = model_data.dropna().copy()

print("Final dataset shape:", model_data.shape)
display(model_data.head())

print("\nFinal missing values:")
print(model_data.isnull().sum())

## 12. Save the Cleaned Dataset

In [ ]:
CLEANED_DATA_PATH = "Cleaned_data.csv"

model_data.to_csv(
    CLEANED_DATA_PATH,
    index=False
)

print(f"Cleaned dataset saved as: {CLEANED_DATA_PATH}")

## 13. Prepare Features and Target

In [ ]:
X = model_data.drop(columns=["price"])
y = model_data["price"]

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

## 14. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

The test set is kept separate from model selection. Cross-validation and hyperparameter tuning are performed only on the training data.

## 15. Build the Preprocessing Pipeline

In [ ]:
categorical_features = ["location"]

numeric_features = [
    "total_sqft",
    "bath",
    "bhk"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "location_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

def create_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", model)
        ]
    )

## 16. Baseline Model Comparison Using 5-Fold Cross-Validation

In [ ]:
baseline_models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(max_iter=10000),
    "Ridge Regression": Ridge()
}

cv_results = []

for model_name, model in baseline_models.items():
    pipeline = create_pipeline(model)

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "Mean CV R2": scores.mean(),
        "Std CV R2": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    by="Mean CV R2",
    ascending=False
)

display(cv_results_df)

## 17. Hyperparameter Tuning with GridSearchCV

### 17.1 Tune Lasso Regression

In [ ]:
lasso_pipeline = create_pipeline(
    Lasso(max_iter=20000)
)

lasso_param_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ]
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

print("Best Lasso Parameters:")
print(lasso_grid.best_params_)

print("\nBest Lasso CV R²:")
print(lasso_grid.best_score_)

### 17.2 Tune Ridge Regression

In [ ]:
ridge_pipeline = create_pipeline(
    Ridge()
)

ridge_param_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
        10,
        50,
        100,
        200
    ]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge Parameters:")
print(ridge_grid.best_params_)

print("\nBest Ridge CV R²:")
print(ridge_grid.best_score_)

## 18. Select the Best Model Based on Cross-Validation

In [ ]:
linear_pipeline = create_pipeline(
    LinearRegression()
)

linear_scores = cross_val_score(
    linear_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

model_selection_results = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "Best CV R2": linear_scores.mean(),
        "Selected Parameters": "Default"
    },
    {
        "Model": "Lasso Regression",
        "Best CV R2": lasso_grid.best_score_,
        "Selected Parameters": str(lasso_grid.best_params_)
    },
    {
        "Model": "Ridge Regression",
        "Best CV R2": ridge_grid.best_score_,
        "Selected Parameters": str(ridge_grid.best_params_)
    }
]).sort_values(
    by="Best CV R2",
    ascending=False
).reset_index(drop=True)

display(model_selection_results)

## 19. Evaluate the Selected Model on the Independent Test Set

In [ ]:
candidate_models = {
    "Linear Regression": (
        linear_scores.mean(),
        linear_pipeline
    ),
    "Lasso Regression": (
        lasso_grid.best_score_,
        lasso_grid.best_estimator_
    ),
    "Ridge Regression": (
        ridge_grid.best_score_,
        ridge_grid.best_estimator_
    )
}

best_model_name = max(
    candidate_models,
    key=lambda name: candidate_models[name][0]
)

best_model = candidate_models[
    best_model_name
][1]

print("Selected model:", best_model_name)

# Fit only after selection, using the complete training set
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

print("\nFinal Test Performance")
print("-" * 35)
print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.4f} Lakhs")
print(f"RMSE     : {rmse:.4f} Lakhs")

## 20. Actual vs Predicted Price Visualisation

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual Price (Lakhs)")
plt.ylabel("Predicted Price (Lakhs)")
plt.tight_layout()
plt.show()

## 21. Save the Final Flask-Ready Model

In [ ]:
MODEL_PATH = "HousePriceModel.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(best_model, file)

print(f"Best model saved successfully as: {MODEL_PATH}")
print(f"Model selected: {best_model_name}")

## 22. Flask Integration Notes

Your Flask application should now load:

```python
HousePriceModel.pkl
```

The prediction input must contain exactly these columns:

```text
location
total_sqft
bath
bhk
```

The saved object is a complete scikit-learn pipeline, so Flask should send raw values directly to `predict()`. The pipeline itself handles location encoding and numerical scaling.


In [ ]:
# Example prediction using the final saved pipeline

sample_input = pd.DataFrame({
    "location": ["Whitefield"],
    "total_sqft": [1200],
    "bath": [2],
    "bhk": [2]
})

sample_prediction = best_model.predict(sample_input)

print(
    f"Predicted Price: {sample_prediction[0]:.2f} Lakhs"
)

# Final Project Workflow

```text
Bengaluru_House_Data.csv
        ↓
Initial Inspection
        ↓
Visual EDA
        ↓
Remove Unnecessary Columns
        ↓
Handle Missing Values
        ↓
Extract BHK
        ↓
Convert total_sqft
        ↓
Explicitly Remove Invalid total_sqft
        ↓
Create price_per_sqft
        ↓
Group Rare Locations
        ↓
Outlier Removal
        ↓
Cleaned_data.csv
        ↓
80/20 Train-Test Split
        ↓
ColumnTransformer
        ↓
OneHotEncoder + StandardScaler
        ↓
5-Fold Cross-Validation
        ↓
GridSearchCV for Lasso and Ridge
        ↓
Best Model Selection
        ↓
Independent Test Evaluation
(R² + MAE + RMSE)
        ↓
HousePriceModel.pkl
        ↓
Flask Web Application
```
